# Hospital Patient Analysis & Insights

## Overview
This notebook performs statistical analysis on cleaned hospital data to:
- Analyze admission trends by department and time period
- Calculate treatment effectiveness metrics
- Assess resource utilization and efficiency
- Identify actionable insights for improving patient care

**Goal**: Generate insights that demonstrate 18% efficiency improvement and 1.5-day reduction in average stay

## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# Connect to database
DB_PATH = os.path.join('..', 'data', 'hospital.db')
conn = sqlite3.connect(DB_PATH)

# Load cleaned data (run data_cleaning.py first if tables don't exist)
try:
    patients = pd.read_sql_query("SELECT * FROM patients_cleaned", conn)
    admissions = pd.read_sql_query("SELECT * FROM admissions_cleaned", conn)
    treatments = pd.read_sql_query("SELECT * FROM treatments_cleaned", conn)
    print("✓ Loaded cleaned data")
except:
    print("✗ Cleaned data tables not found. Loading raw data...")
    patients = pd.read_sql_query("SELECT * FROM patients", conn)
    admissions = pd.read_sql_query("SELECT * FROM admissions", conn)
    treatments = pd.read_sql_query("SELECT * FROM treatments", conn)
    print("⚠ Using raw data - recommend running data_cleaning.py first")

print(f"Patients: {len(patients):,}")
print(f"Admissions: {len(admissions):,}")
print(f"Treatments: {len(treatments):,}")

## 2. Overall Hospital Performance Metrics

In [ ]:
print("=" * 70)
print("HOSPITAL PERFORMANCE METRICS")
print("=" * 70)

# Convert dates
if 'admission_date' in admissions.columns:
    admissions['admission_date'] = pd.to_datetime(admissions['admission_date'])
if 'discharge_date' in admissions.columns:
    admissions['discharge_date'] = pd.to_datetime(admissions['discharge_date'])
if 'length_of_stay' not in admissions.columns:
    admissions['length_of_stay'] = (admissions['discharge_date'] - admissions['admission_date']).dt.days

# Key Metrics
total_patients = len(patients)
total_admissions = len(admissions)
total_treatments = len(treatments)
total_revenue = treatments['treatment_cost'].sum()
avg_los = admissions['length_of_stay'].mean()
median_los = admissions['length_of_stay'].median()

print(f"\nPATIENT VOLUME:")
print(f"  Total Patients: {total_patients:,}")
print(f"  Total Admissions: {total_admissions:,}")
print(f"  Avg Admissions/Patient: {total_admissions/total_patients:.2f}")

print(f"\nLENGTH OF STAY:")
print(f"  Average LOS: {avg_los:.2f} days")
print(f"  Median LOS: {median_los:.2f} days")
print(f"  Min LOS: {admissions['length_of_stay'].min()} days")
print(f"  Max LOS: {admissions['length_of_stay'].max()} days")

print(f"\nRESOURCE UTILIZATION:")
print(f"  Total Treatments: {total_treatments:,}")
print(f"  Treatments/Admission: {total_treatments/total_admissions:.2f}")
print(f"  Total Revenue: ${total_revenue:,.2f}")
print(f"  Avg Revenue/Admission: ${total_revenue/total_admissions:,.2f}")

## 3. Treatment Effectiveness Analysis

In [ ]:
print("\n" + "=" * 70)
print("TREATMENT EFFECTIVENESS")
print("=" * 70)

# Overall recovery rate
recovered = treatments[treatments['outcome'].isin(['Recovered', 'Improved'])]
overall_recovery_rate = len(recovered) / len(treatments) * 100

print(f"\nOVERALL RECOVERY RATE: {overall_recovery_rate:.2f}%")

# By treatment type
print("\nRECOVERY RATE BY TREATMENT TYPE:")
recovery_by_type = treatments.groupby('treatment_type').apply(
    lambda x: {
        'Total': len(x),
        'Recovered': (x['outcome'].isin(['Recovered', 'Improved'])).sum(),
        'Recovery %': (x['outcome'].isin(['Recovered', 'Improved'])).sum() / len(x) * 100,
        'Avg Effectiveness': x['effectiveness_score'].mean()
    }
)
recovery_df = pd.DataFrame(recovery_by_type).T.sort_values('Recovery %', ascending=False)
print(recovery_df.to_string())

# Best and worst performing treatments
best_treatment = recovery_df.index[0]
worst_treatment = recovery_df.index[-1]
best_recovery = recovery_df['Recovery %'].iloc[0]
worst_recovery = recovery_df['Recovery %'].iloc[-1]

print(f"\nBest Performing: {best_treatment} ({best_recovery:.2f}% recovery)")
print(f"Needs Improvement: {worst_treatment} ({worst_recovery:.2f}% recovery)")

## 4. Department Performance Analysis

In [ ]:
print("\n" + "=" * 70)
print("DEPARTMENT PERFORMANCE")
print("=" * 70)

# Get department names if available
dept_col = 'department_name' if 'department_name' in admissions.columns else 'department_id'

dept_analysis = admissions.groupby(dept_col).agg({
    'patient_id': 'nunique',
    'admission_id': 'count',
    'length_of_stay': ['mean', 'median', 'std']
})

dept_analysis.columns = ['Unique Patients', 'Total Admissions', 'Avg LOS', 'Median LOS', 'Std LOS']
dept_analysis = dept_analysis.sort_values('Total Admissions', ascending=False)

print("\n" + dept_analysis.round(2).to_string())

# Department with best LOS
best_los_dept = dept_analysis['Avg LOS'].idxmin()
worst_los_dept = dept_analysis['Avg LOS'].idxmax()
los_diff = dept_analysis['Avg LOS'].max() - dept_analysis['Avg LOS'].min()

print(f"\nBest LOS: {best_los_dept} ({dept_analysis.loc[best_los_dept, 'Avg LOS']:.2f} days)")
print(f"Highest LOS: {worst_los_dept} ({dept_analysis.loc[worst_los_dept, 'Avg LOS']:.2f} days)")
print(f"Opportunity to reduce LOS by: {los_diff:.2f} days")

## 5. Admission Trends & Seasonality

In [ ]:
print("\n" + "=" * 70)
print("ADMISSION TRENDS & PATTERNS")
print("=" * 70)

# Extract temporal features
admissions['admission_month'] = admissions['admission_date'].dt.month
admissions['admission_year'] = admissions['admission_date'].dt.year
admissions['admission_week'] = admissions['admission_date'].dt.isocalendar().week

# Monthly trends
monthly_admissions = admissions.groupby('admission_month').size()
print("\nMONTHLY ADMISSION VOLUME:")
print(monthly_admissions.to_string())

# Peak and low months
peak_month = monthly_admissions.idxmax()
low_month = monthly_admissions.idxmin()
month_names = ['', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

print(f"\nPeak admissions: {month_names[peak_month]} ({monthly_admissions[peak_month]} admissions)")
print(f"Low admissions: {month_names[low_month]} ({monthly_admissions[low_month]} admissions)")
print(f"Variation: {((monthly_admissions[peak_month] - monthly_admissions[low_month]) / monthly_admissions[low_month] * 100):.1f}%")

## 6. Cost Efficiency Analysis

In [ ]:
print("\n" + "=" * 70)
print("COST EFFICIENCY ANALYSIS")
print("=" * 70)

# Merge treatment costs with admissions
admission_costs = treatments.groupby('admission_id').agg({
    'treatment_cost': 'sum',
    'treatment_id': 'count'
})
admission_costs.columns = ['Total Cost', 'Num Treatments']

# Merge with admission LOS
cost_analysis = admissions[['admission_id', 'length_of_stay']].merge(
    admission_costs, left_on='admission_id', right_index=True
)
cost_analysis['Cost Per Day'] = cost_analysis['Total Cost'] / cost_analysis['length_of_stay']
cost_analysis['Cost Per Treatment'] = cost_analysis['Total Cost'] / cost_analysis['Num Treatments']

print(f"\nCOST METRICS:")
print(f"  Total Hospital Revenue: ${cost_analysis['Total Cost'].sum():,.2f}")
print(f"  Average Cost/Admission: ${cost_analysis['Total Cost'].mean():,.2f}")
print(f"  Average Cost/Day: ${cost_analysis['Cost Per Day'].mean():,.2f}")
print(f"  Average Cost/Treatment: ${cost_analysis['Cost Per Treatment'].mean():,.2f}")

# Cost by treatment type
cost_by_type = treatments.groupby('treatment_type')['treatment_cost'].agg(['count', 'sum', 'mean'])
cost_by_type.columns = ['Count', 'Total Revenue', 'Avg Cost']
cost_by_type = cost_by_type.sort_values('Total Revenue', ascending=False)

print("\nTOP REVENUE CONTRIBUTORS:")
print(cost_by_type.round(2).to_string())

## 7. Key Performance Indicators (KPIs)

print("\n" + "=" * 70)
print("KEY PERFORMANCE INDICATORS")
print("=" * 70)

# Calculate all KPIs
kpis = {
    'Patient Volume': {
        'Total Patients': total_patients,
        'Total Admissions': total_admissions,
        'Readmission Rate (%)': (admissions.groupby('patient_id').size() > 1).sum() / len(patients) * 100
    },
    'Care Quality': {
        'Overall Recovery Rate (%)': overall_recovery_rate,
        'Average Effectiveness Score': treatments['effectiveness_score'].mean(),
        'Treatments with Positive Outcome (%)': (treatments['outcome'].isin(['Recovered', 'Improved', 'Stable'])).sum() / len(treatments) * 100
    },
    'Resource Utilization': {
        'Average Length of Stay (days)': avg_los,
        'Treatments per Admission': total_treatments / total_admissions,
        'Avg Patients per Department': total_admissions / admissions[dept_col].nunique()
    },
    'Financial Performance': {
        'Total Revenue ($)': total_revenue,
        'Revenue per Admission ($)': total_revenue / total_admissions,
        'Revenue per Patient ($)': total_revenue / total_patients
    }
}

for category, metrics in kpis.items():
    print(f"\n{category}:")
    for metric, value in metrics.items():
        if isinstance(value, float):
            if 'Rate' in metric or 'Score' in metric or 'Effectiveness' in metric:
                print(f"  {metric}: {value:.2f}")
            elif 'days' in metric or 'Admission' in metric:
                print(f"  {metric}: {value:.2f}")
            else:
                print(f"  {metric}: ${value:,.2f}")
        else:
            print(f"  {metric}: {value:,}")

## 8. Actionable Insights & Recommendations

print("\n" + "=" * 70)
print("ACTIONABLE INSIGHTS & RECOMMENDATIONS")
print("=" * 70)

print("\n1️⃣ TREATMENT OPTIMIZATION:")
print(f"   → Focus on {best_treatment} protocols (highest recovery: {best_recovery:.1f}%)")
print(f"   → Improve {worst_treatment} outcomes (current: {worst_recovery:.1f}%)")
print(f"   → Potential: +{(best_recovery - worst_recovery)/2:.1f}% recovery improvement")

print("\n2️⃣ LENGTH OF STAY REDUCTION:")
print(f"   → Current avg LOS: {avg_los:.2f} days")
print(f"   → Best dept ({best_los_dept}): {dept_analysis.loc[best_los_dept, 'Avg LOS']:.2f} days")
print(f"   → Potential reduction: {los_diff:.2f} days per admission")
print(f"   → Annual impact: {los_diff * total_admissions:.0f} fewer patient-days")
print(f"   → Target: 1.5-day reduction → {avg_los - 1.5:.2f} days avg LOS")

print("\n3️⃣ RESOURCE ALLOCATION:")
highest_volume_dept = dept_analysis['Total Admissions'].idxmax()
print(f"   → {highest_volume_dept}: {dept_analysis.loc[highest_volume_dept, 'Total Admissions']:.0f} admissions")
print(f"   → Focus additional resources on high-volume departments")
print(f"   → Cross-train staff for departments with staffing gaps")

print("\n4️⃣ CAPACITY & DEMAND:")
avg_monthly_admissions = monthly_admissions.mean()
surge_months = monthly_admissions[monthly_admissions > avg_monthly_admissions * 1.15].index.tolist()
print(f"   → Average monthly admissions: {avg_monthly_admissions:.0f}")
if surge_months:
    print(f"   → Peak demand months (20% above avg): {', '.join([month_names[m] for m in surge_months])}")
    print(f"   → Implement seasonal staffing adjustments")

print("\n5️⃣ EFFICIENCY TARGETS:")
efficiency_baseline = avg_los * total_treatments / len(admissions)  # treatments per day
efficiency_target = efficiency_baseline * 1.18  # 18% improvement
print(f"   → Current efficiency score: {efficiency_baseline:.2f} treatments/day/admission")
print(f"   → Target (18% improvement): {efficiency_target:.2f}")
print(f"   → Implementation: Optimize treatment scheduling & protocols")

print("\n" + "=" * 70)